# SG-Aware Reward — Real vs SD vs SD-perturbed

**Why this redesign.** The previous notebook scored a single SD image against an original graph and a perturbed graph. That only tested whether the reward could *detect a graph change* — it never tested whether **SD actually honors the perturbation**. If SD ignores `rabbit:red→green` and still draws a red rabbit, the reward should not drop — but the previous design called that a failure.

**This notebook compares three images per example:**

| Image | Source | Role |
|---|---|---|
| `real`     | A real photograph we drop in `notebooks/real_images/ex_NN.jpg`. Falls back to SD with a separate seed if absent. | Ground-truth reference for what the original graph *should* look like. |
| `SD_orig`  | SD1.5 generated from the original prompt, seed=i. | Tests SD's fidelity to the original prompt. |
| `SD_pert`  | SD1.5 generated from the perturbed prompt, seed=i. Different image per perturbation type. | Tests whether SD honored the graph perturbation. |

**SG-Aware uses the predefined graph.** No detection, no graph extraction. The graph is human-authored ground truth; SG-Aware just scores each node/edge against the image via BLIP-ITM with antonym contrast.

**Dataset.** Structured graphs in FIBO style (Bria-AI, arXiv 2511.06876) — `objects` + per-object attributes + binary relations. FIBO's training set isn't public, so we hand-author 5 graphs in their format.

## 1. Imports + flags

In [ ]:
import sys, math, random, tempfile, os, io
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt
import networkx as nx
import urllib.request
from IPython.display import display

REPO = Path('..').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device} | numpy: {np.__version__} | pandas: {pd.__version__}')
random.seed(0); np.random.seed(0); torch.manual_seed(0)

INCLUDE_SLOW_METRICS = False   # set True if t2v-metrics available
REAL_IMG_DIR = Path('real_images'); REAL_IMG_DIR.mkdir(exist_ok=True)
SD_IMG_DIR   = Path('sd_images_real_demo'); SD_IMG_DIR.mkdir(exist_ok=True)

## 2. The 4 examples — taken from FIBO's released outputs

The 4 examples come directly from `Bria-AI/FIBO/examples/outputs/`. Each example is a (reference image, structured caption) pair released by the FIBO authors; we extract a simplified `SceneGraph` (objects + attributes + binary relations) from FIBO's structured JSON.

FIBO's training corpus is **not public** — only these example outputs are released — so 4 is what we have. The notebook downloads each PNG directly from FIBO's GitHub raw URL; you can override any of them by dropping a replacement at `notebooks/real_images/ex_NN.jpg`.


In [ ]:
from dpok_scene_reward import SceneGraph
from perturb_graph import PERTURBATIONS, stringify_graph

# Four examples taken directly from FIBO's released examples/outputs/ folder
# (Bria-AI, https://github.com/Bria-AI/FIBO).  Each has a public PNG + structured
# JSON in that repo; we extract the SceneGraph fields from FIBO's JSON.
#
# FIBO's distinct released outputs (no public training set is available):
#   - refine_original.{png,json}  zebra + red fire hydrant  (photograph)
#   - refine_edited.{png,json}    elephant + red fire hydrant  (cat→dog→elephant edit)
#   - inspire.{png,json}          zebra + balloons + fire hydrant  (3D render)
#   - generate.{png,json}         brown dog in park  (single object)

FIBO_RAW = 'https://raw.githubusercontent.com/Bria-AI/FIBO/main/examples/outputs'

EXAMPLES = [
    # 0 — FIBO refine_original.json: zebra + red fire hydrant (photo)
    dict(prompt=('A realistic image features a zebra standing on a concrete '
                 'sidewalk next to a red fire hydrant'),
         graph=SceneGraph(['zebra', 'fire hydrant'],
                          [('zebra', 'striped'), ('fire hydrant', 'red')],
                          [('zebra', 'next to', 'fire hydrant')]),
         real_url=f'{FIBO_RAW}/refine_original.png',
         fibo_name='refine_original'),
    # 1 — FIBO refine_edited.json: elephant + red fire hydrant (the "edit" output)
    dict(prompt=('A realistic image features a gray elephant standing on a '
                 'concrete sidewalk next to a red fire hydrant'),
         graph=SceneGraph(['elephant', 'fire hydrant'],
                          [('elephant', 'gray'), ('fire hydrant', 'red')],
                          [('elephant', 'next to', 'fire hydrant')]),
         real_url=f'{FIBO_RAW}/refine_edited.png',
         fibo_name='refine_edited'),
    # 2 — FIBO inspire.json: zebra + balloons + fire hydrant (3D render)
    dict(prompt=('A whimsical 3D render of a striped zebra floating with '
                 'pastel balloons above a red fire hydrant'),
         graph=SceneGraph(['zebra', 'balloons', 'fire hydrant'],
                          [('zebra', 'striped'),
                           ('balloons', 'pastel'),
                           ('fire hydrant', 'red')],
                          [('balloons', 'above', 'zebra'),
                           ('fire hydrant', 'left of', 'zebra')]),
         real_url=f'{FIBO_RAW}/inspire.png',
         fibo_name='inspire'),
    # 3 — FIBO generate.json: brown dog (single object — spatial perturbation will no-op)
    dict(prompt='A light brown medium-sized dog playing in a grassy park',
         graph=SceneGraph(['dog'],
                          [('dog', 'brown')],
                          []),
         real_url=f'{FIBO_RAW}/generate.png',
         fibo_name='generate'),
]

print(f'{len(EXAMPLES)} examples — directly from FIBO examples/outputs/.\n')
for i, e in enumerate(EXAMPLES):
    print(f"  ex{i} ({e['fibo_name']}): {e['prompt']}")
    print(f"         graph: {stringify_graph(e['graph'])}")


## 3. Load the 5 real reference images

Order: local `real_images/ex_NN.jpg` → URL fetch → SD with seed `(i+500)` as a stand-in. Each example's `'real_image'` ends up populated either way; the per-image `'real_source'` records which path was used (matters for slide honesty).

In [ ]:
def fetch_url(url, dst, timeout=15):
    req = urllib.request.Request(url, headers={'User-Agent': 'sg_demo/1.0'})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        data = r.read()
    Image.open(io.BytesIO(data)).convert('RGB').save(dst)

for i, ex in enumerate(EXAMPLES):
    local = REAL_IMG_DIR / f'ex_{i:02d}.jpg'
    if local.exists():
        ex['real_image'] = Image.open(local).convert('RGB')
        ex['real_source'] = 'user-provided'
    else:
        try:
            fetch_url(ex['real_url'], local)
            ex['real_image'] = Image.open(local).convert('RGB')
            ex['real_source'] = 'pexels-url'
        except Exception as e:
            print(f'  ⚠ ex{i}: download failed ({type(e).__name__}); will use SD stand-in.')
            ex['real_image'] = None
            ex['real_source'] = 'sd-standin'
    print(f"  ex{i}: {ex['real_source']:<14s}  {ex['prompt']}")

## 4. Generate SD-original (and SD-stand-in for any failed real fetch)

`SD_orig`  ← SD with seed `i` from the original prompt.
`SD_real`  ← only used if the real URL failed; SD with seed `i+500` from the same prompt, treated as a stand-in for the real image (and flagged as such in plots).

In [ ]:
from diffusers import StableDiffusionPipeline

print('Loading SD1.5 ...')
pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=torch.float16 if device=='cuda' else torch.float32,
    safety_checker=None,
).to(device)
pipe.set_progress_bar_config(disable=True)

def sd_gen(prompt, seed, num_steps=30):
    gen = torch.Generator(device=device).manual_seed(seed)
    return pipe(prompt, num_inference_steps=num_steps, guidance_scale=7.5,
                generator=gen).images[0]

def gen_or_load(prompt, idx, kind, seed):
    path = SD_IMG_DIR / f'{kind}_{idx:02d}.png'
    if path.exists():
        return Image.open(path).convert('RGB')
    img = sd_gen(prompt, seed)
    img.save(path)
    return img

for i, ex in enumerate(EXAMPLES):
    ex['sd_orig'] = gen_or_load(ex['prompt'], i, 'sd_orig', seed=i)
    if ex['real_image'] is None:
        ex['real_image'] = gen_or_load(ex['prompt'], i, 'sd_standin', seed=i + 500)
    print(f"  ex{i} done")
print('SD-orig images ready.')

## 5. Visualize: real | SD-orig | graph

One row per example: real photo, SD-orig, and the structured graph drawn as a small node-edge diagram.

In [ ]:
def draw_graph(g, ax, title=''):
    G = nx.DiGraph()
    attr_map = {o: [] for o in g.objects}
    for o, a in g.attributes:
        attr_map[o].append(a)
    for obj in g.objects:
        label = f"{' '.join(attr_map[obj])}\n{obj}".strip()
        G.add_node(obj, label=label)
    for s, r, o in g.relations:
        G.add_edge(s, o, label=r)
    pos = nx.spring_layout(G, seed=1, k=2.0) if len(G.nodes) > 1 else {list(G.nodes)[0]: (0,0)}
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color='lightblue', node_size=3000, edgecolors='black')
    nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowsize=20, edge_color='gray')
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=9,
                            labels={n: G.nodes[n]['label'] for n in G.nodes})
    nx.draw_networkx_edge_labels(G, pos, ax=ax, font_size=8,
                                 edge_labels={(s,o): r for s,r,o in g.relations})
    ax.set_title(title, fontsize=10)
    ax.axis('off')

fig, axes = plt.subplots(len(EXAMPLES), 3, figsize=(15, 4*len(EXAMPLES)))
for i, ex in enumerate(EXAMPLES):
    axes[i,0].imshow(ex['real_image']); axes[i,0].axis('off')
    axes[i,0].set_title(f"REAL ({ex['real_source']})", fontsize=10)
    axes[i,1].imshow(ex['sd_orig']); axes[i,1].axis('off')
    axes[i,1].set_title('SD_orig (seed=i)', fontsize=10)
    draw_graph(ex['graph'], axes[i,2], title=f"Graph\n{ex['prompt']}")
plt.suptitle('Reference photos, SD-original, and structured graphs', fontsize=14, y=1.005)
plt.tight_layout()
plt.savefig('real_vs_sd_orig.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Perturb each graph + generate one SD-perturbed image per perturbation

For each example × each perturbation: produce a perturbed graph + a natural-language prompt that reflects the perturbation, then SD-generate that prompt at the same seed `i` (only the prompt differs from `SD_orig`).

In [ ]:
def prompt_from_graph(g):
    attr_map = {}
    for o, a in g.attributes:
        attr_map.setdefault(o, []).append(a)
    parts = [f"a {' '.join(attr_map.get(o, []))} {o}".strip().replace('  ',' ') for o in g.objects]
    obj_str = ' and '.join(parts)
    if not g.relations:
        return obj_str
    s, r, o = g.relations[0]
    return f"{obj_str}, where the {s} is {r} the {o}"

rng = random.Random(0)
for i, ex in enumerate(EXAMPLES):
    ex['perturbations'] = {}
    for pname, pfn in PERTURBATIONS.items():
        new_g, label = pfn(ex['graph'], rng)
        new_prompt = prompt_from_graph(new_g)
        img = gen_or_load(new_prompt, i, f'sd_pert_{pname}', seed=i)
        ex['perturbations'][pname] = dict(graph=new_g, label=label, prompt=new_prompt, image=img)
    print(f'  ex{i} ({ex["prompt"][:35]}...) — 4 perturbations generated')

# free VRAM before scorer loads
del pipe
if device == 'cuda': torch.cuda.empty_cache()
print('All SD-perturbed images ready.')

## 7. Visualize: per-example perturbation grid

For one chosen example: show real, SD-orig, and 4 SD-perturbed images, each with the graph that produced it.

In [ ]:
def show_full_row(idx):
    ex = EXAMPLES[idx]
    fig, axes = plt.subplots(2, 6, figsize=(28, 9),
                             gridspec_kw={'height_ratios': [3, 2]})
    # top row: 6 images
    axes[0,0].imshow(ex['real_image']); axes[0,0].axis('off')
    axes[0,0].set_title(f"REAL ({ex['real_source']})", fontsize=10)
    axes[0,1].imshow(ex['sd_orig']); axes[0,1].axis('off')
    axes[0,1].set_title(f"SD_orig\nseed={idx}", fontsize=10)
    for j, (pname, info) in enumerate(ex['perturbations'].items(), start=2):
        axes[0,j].imshow(info['image']); axes[0,j].axis('off')
        axes[0,j].set_title(f"SD_{pname}\n({info['label']})", fontsize=9)
    # bottom row: corresponding graphs
    draw_graph(ex['graph'], axes[1,0], title='original graph')
    draw_graph(ex['graph'], axes[1,1], title='original graph')
    for j, (pname, info) in enumerate(ex['perturbations'].items(), start=2):
        draw_graph(info['graph'], axes[1,j], title=f'{pname}')
    plt.suptitle(f"Example {idx}: {ex['prompt']}", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'perturbation_grid_ex{idx:02d}.png', dpi=110, bbox_inches='tight')
    plt.show()

show_full_row(0)

In [ ]:
show_full_row(2)

## 8. Load scorers (4 fast, BLIP-only SG-Aware)

In [ ]:
import torch.nn.functional as F
import open_clip
import ImageReward as RM
from dpok_scene_reward import BLIPRewardModel
from dpok_sg_reward import SGAwareReward, SGAwareConfig

print('Loading CLIP ...')
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
clip_model = clip_model.to(device).eval()
for p in clip_model.parameters(): p.requires_grad_(False)
clip_tok = open_clip.get_tokenizer('ViT-L-14')

@torch.no_grad()
def clip_score(img, prompt):
    t = clip_preprocess(img).unsqueeze(0).to(device)
    iv = F.normalize(clip_model.encode_image(t), dim=-1)
    tv = F.normalize(clip_model.encode_text(clip_tok([prompt]).to(device)), dim=-1)
    return float((iv * tv).sum().cpu())

print('Loading ImageReward ...')
ir_model = RM.load('ImageReward-v1.0', device=device)
@torch.no_grad()
def ir_score(img, prompt):
    tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    img.convert('RGB').save(tmp.name); tmp.close()
    try:
        return float(ir_model.score(prompt, tmp.name))
    finally:
        os.unlink(tmp.name)

print('Loading BLIP-ITM ...')
blip_reward = BLIPRewardModel(device=device, baseline=0.0, scale=1.0)
def blip_hierarchical(img, g):
    pieces = ([f'a photo of a {o}' for o in g.objects] +
              [f'a {a} {o}' for o, a in g.attributes] +
              [f'{s} {r} {o}' for s, r, o in g.relations])
    if not pieces: return 0.0
    return sum(float(blip_reward.compute(img, p).cpu()) for p in pieces) / len(pieces)

print('Loading SG-Aware (uses the predefined graph directly) ...')
sg = SGAwareReward(device=device, config=SGAwareConfig(),
                   image_reward_model=ir_model, blip_reward_model=blip_reward)
def sg_score(img, g):
    return sg.score(img, g)[0]

REWARDS = {
    'CLIPScore':         lambda img, g: clip_score(img, stringify_graph(g)),
    'ImageReward':       lambda img, g: ir_score(img, stringify_graph(g)),
    'BLIP hierarchical': lambda img, g: blip_hierarchical(img, g),
    'SG-Aware':          sg_score,
}

# Optional slow scorers
if INCLUDE_SLOW_METRICS:
    try:
        from dpok_eval_metrics import VQAScorer, DSGScorer
        _vqa = VQAScorer(device=device); _dsg = DSGScorer(vqa_scorer=_vqa)
        REWARDS['VQAScore'] = lambda img, g: _vqa.score(img, stringify_graph(g))
        REWARDS['DSGScore'] = lambda img, g: _dsg.score(img, stringify_graph(g))[0]
        print('  VQA + DSG enabled.')
    except Exception as e:
        print(f'  ⚠ Slow scorers unavailable ({type(e).__name__}: {e}). Skipping.')

print(f'\nReady: {list(REWARDS.keys())}')

## 9. Score every (image, graph) combination

Per example × per perturbation, we score 3 images (real, SD_orig, SD_pert) against 2 graphs (original, perturbed). That's 6 scores per (example, perturbation, reward) row.

In [ ]:
rows = []
for i, ex in enumerate(EXAMPLES):
    G = ex['graph']
    for pname, info in ex['perturbations'].items():
        Gp = info['graph']
        images = {
            'real':    ex['real_image'],
            'SD_orig': ex['sd_orig'],
            'SD_pert': info['image'],
        }
        for img_kind, img in images.items():
            for reward_name, fn in REWARDS.items():
                s_orig = fn(img, G)
                s_pert = fn(img, Gp)
                rows.append(dict(
                    example=i, prompt=ex['prompt'],
                    perturbation=pname, label=info['label'],
                    reward=reward_name,
                    image_kind=img_kind,
                    score_orig_graph=s_orig,
                    score_pert_graph=s_pert,
                    graph_delta=s_orig - s_pert,        # +→ image fits orig more than pert (good for real, SD_orig)
                ))
    print(f'  ex{i} scored')

df = pd.DataFrame(rows)
df.to_csv('sg_real_full.csv', index=False)
print(f'\n{len(df)} rows.')
df.head(12)

## 10. Headline table — graph_delta per (reward × image_kind)

`graph_delta = score(image, orig_graph) − score(image, perturbed_graph)`. 

- **Real / SD_orig** should give **positive** delta (these images fit the original graph, not the perturbed one).
- **SD_pert** delta tells us whether SD honored the perturbation: positive means SD ignored the perturbation (image still fits original graph), near zero / negative means SD followed the perturbation (image fits perturbed graph better).

In [ ]:
headline = df.pivot_table(
    index='reward', columns='image_kind',
    values='graph_delta', aggfunc='mean',
)[['real', 'SD_orig', 'SD_pert']]
headline = headline.reindex(list(REWARDS.keys()))
display(headline.style.format('{:+.3f}').background_gradient(cmap='RdBu_r', axis=None))
headline.to_csv('sg_real_headline.csv')

print("\nReading guide:")
print("  real & SD_orig columns: ↑ positive = reward sees the original-graph fit")
print("  SD_pert column        : ↓ closer to zero / negative = SD honored the perturbation")
print("  larger gap between (real/SD_orig) and (SD_pert) = reward is graph-sensitive")

## 11. Per-perturbation breakdown

Same `graph_delta`, but split by perturbation type. Shows which perturbations SD honors and which it ignores.

In [ ]:
for kind in ['real', 'SD_orig', 'SD_pert']:
    sub = df[df.image_kind == kind]
    pv = sub.pivot_table(index='reward', columns='perturbation',
                         values='graph_delta', aggfunc='mean')
    pv = pv[['remove_object','swap_attribute','swap_spatial','swap_noun']]
    pv = pv.reindex(list(REWARDS.keys()))
    print(f'\n=== image_kind = {kind} — mean graph_delta ===')
    display(pv.style.format('{:+.3f}').background_gradient(cmap='RdBu_r', axis=None))

## 12. Diagnostic — did SD honor the perturbation?

Compare `score(SD_orig, orig_G)` (high baseline) against `score(SD_pert, orig_G)` (should drop if SD changed the image). The gap shows how much SD shifted the image away from the original concept.

In [ ]:
sd_orig_scores = (df[df.image_kind == 'SD_orig']
                  .groupby(['reward','perturbation'])['score_orig_graph'].mean().unstack())
sd_pert_scores = (df[df.image_kind == 'SD_pert']
                  .groupby(['reward','perturbation'])['score_orig_graph'].mean().unstack())
sd_shift = sd_orig_scores - sd_pert_scores   # positive = SD changed the image away from orig graph
sd_shift = sd_shift[['remove_object','swap_attribute','swap_spatial','swap_noun']]
sd_shift = sd_shift.reindex(list(REWARDS.keys()))
print('SD shift = score(SD_orig vs orig_G) − score(SD_pert vs orig_G)')
print('Larger → SD honored the perturbation more strongly (image drifted from orig).')
display(sd_shift.style.format('{:+.3f}').background_gradient(cmap='Blues', axis=None))
sd_shift.to_csv('sg_real_sd_shift.csv')

## 13. Per-example score view (one row per example × perturbation)

Inspect specific (example, perturbation) pairs.

In [ ]:
def show_scores(idx, pname='swap_attribute'):
    sub = df[(df.example == idx) & (df.perturbation == pname)]
    pv = sub.pivot_table(
        index='reward', columns='image_kind',
        values=['score_orig_graph','score_pert_graph'],
    )
    print(f"=== ex{idx} — {pname} ({sub.iloc[0]['label']}) ===")
    print(f"prompt: {EXAMPLES[idx]['prompt']}")
    display(pv.style.format('{:+.3f}'))

show_scores(0, 'swap_attribute')
show_scores(0, 'swap_spatial')
show_scores(2, 'remove_object')

## 14. Files this notebook produced

- `notebooks/real_vs_sd_orig.png` — slide: reference photos vs SD outputs (with graphs)
- `notebooks/perturbation_grid_exNN.png` — slide: one example × 4 perturbations with images + graphs
- `notebooks/sg_real_headline.csv` — headline graph-delta table
- `notebooks/sg_real_sd_shift.csv` — SD-fidelity-to-perturbation table
- `notebooks/sg_real_full.csv` — all 5 × 4 × N_rewards × 3 images rows
- `notebooks/sd_images_real_demo/sd_orig_NN.png`, `sd_pert_*_NN.png` — cached SD outputs
- `notebooks/real_images/ex_NN.jpg` — cached real reference photos (overwritable)